# Tensor & Transformer Manipulation Lab
## Practical techniques, best practices, and runnable PyTorch examples

This notebook is a hands-on guide to manipulating tensors and Transformer representations, from fundamental tensor operations through attention, residual streams, activation interventions, weight transformations, compression, and interpretability.

**Focus:** PyTorch first, with NumPy for small reference calculations. Examples use synthetic data and small models so the core notebook can run on CPU.

### Learning path

1. Tensor fundamentals, shapes, strides, views, and memory
2. Indexing, broadcasting, reductions, and tensor contractions
3. Autograd, gradients, and safe in-place operations
4. Linear layers, embeddings, normalization, and activations
5. Convolution and sequence operations
6. Transformer tensors: Q, K, V, attention masks, and multi-head attention
7. Residual streams, hidden states, and hooks
8. Activation interventions and feature steering
9. Weight manipulation, initialization, and model merging
10. Tensor decomposition, sparsity, pruning, and quantization
11. Performance, numerical stability, testing, and reproducibility

> **Important distinction:** “Tensor manipulation” can mean ordinary shape/value operations, changing intermediate activations in a running model, or changing learned model weights. Those are related but different activities. This lab separates them.

## 0. Environment and reproducibility

Recommended packages:

```bash
python -m pip install torch numpy matplotlib
```

For later model-specific experiments you may additionally use `transformers`, `accelerate`, and `safetensors`. They are not required for the core examples.

Best practices:
- Record library versions and random seeds.
- Start with small tensors and assert expected shapes.
- Prefer explicit dimensions (`dim=-1`) over relying on defaults.
- Avoid unnecessary copies and in-place mutation of tensors used by autograd.
- Test numerical behavior on CPU before scaling to GPU.

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Device:", device)

# Part I — Tensor fundamentals

## 1. Tensor ranks, shapes, and dimensions

A tensor is a multi-dimensional array. In deep learning, shape conventions matter as much as the values.

Common conventions:
- Scalar: `[]`
- Vector: `[features]`
- Matrix: `[rows, columns]`
- Image batch: `[batch, channels, height, width]`
- Sequence batch: `[batch, sequence, features]`
- Attention scores: `[batch, heads, query_sequence, key_sequence]`

Always write down the shape contract for a function.

In [ ]:
scalar = torch.tensor(3.14)
vector = torch.tensor([1., 2., 3.])
matrix = torch.randn(2, 3)
sequence_batch = torch.randn(4, 10, 32)  # batch=4, tokens=10, hidden=32
image_batch = torch.randn(8, 3, 64, 64)

for name, tensor in {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "sequence_batch": sequence_batch,
    "image_batch": image_batch,
}.items():
    print(f"{name:16} shape={tuple(tensor.shape)}, ndim={tensor.ndim}, dtype={tensor.dtype}")

### Shape assertions as executable documentation

In [ ]:
def validate_hidden_states(hidden: torch.Tensor, hidden_size: int) -> None:
    assert hidden.ndim == 3, "Expected [batch, sequence, hidden]"
    assert hidden.shape[-1] == hidden_size
    assert hidden.is_floating_point()

validate_hidden_states(sequence_batch, hidden_size=32)
print("Shape contract passed.")

## 2. Reshape, view, transpose, permute, flatten, and contiguous

These operations are not interchangeable:

- `view()` returns a view when the tensor's strides permit it.
- `reshape()` returns a view when possible, otherwise may copy.
- `transpose()` swaps two dimensions.
- `permute()` reorders any number of dimensions.
- `flatten()` combines dimensions.
- `contiguous()` creates a contiguous layout if needed.

A tensor can have the expected shape but a non-contiguous memory layout. This matters for performance and for `view()`.

In [ ]:
x = torch.arange(2 * 3 * 4).reshape(2, 3, 4)
print("x:", x.shape, "stride:", x.stride(), "contiguous:", x.is_contiguous())

x_t = x.transpose(1, 2)
print("transpose:", x_t.shape, "stride:", x_t.stride(), "contiguous:", x_t.is_contiguous())

# reshape is robust when a view is not possible.
flat = x_t.reshape(2, -1)
print("reshape:", flat.shape)

# If view is needed, make the layout contiguous first.
flat_view = x_t.contiguous().view(2, -1)
print("contiguous + view:", flat_view.shape)

**Best practice:** use `reshape()` unless you specifically need view semantics. Use `contiguous()` only when required; it can allocate and copy memory.

## 3. Indexing, slicing, masking, and gather

Use basic slicing for contiguous ranges, boolean masks for filtering, and `gather` when each batch row selects different indices.

In [ ]:
x = torch.arange(2 * 5 * 3).reshape(2, 5, 3)

first_two_tokens = x[:, :2, :]
last_token = x[:, -1, :]
mask = torch.tensor([
    [True, True, False, False, False],
    [True, False, True, False, True],
])

selected = x[mask]  # flattened selection across first two dimensions

# Gather one token index per batch item.
indices = torch.tensor([1, 3]).view(2, 1, 1).expand(-1, 1, 3)
chosen = torch.gather(x, dim=1, index=indices)

print("first two:", first_two_tokens.shape)
print("last:", last_token.shape)
print("masked:", selected.shape)
print("gathered:", chosen.shape)

## 4. Broadcasting and elementwise operations

Broadcasting aligns dimensions from the right. Dimensions are compatible when they are equal or one of them is 1.

A common Transformer example is applying a per-feature scale and bias to `[batch, sequence, hidden]`.

In [ ]:
hidden = torch.randn(2, 5, 8)
scale = torch.randn(8)
bias = torch.randn(8)

adjusted = hidden * scale + bias
print(hidden.shape, scale.shape, adjusted.shape)

# Explicit dimensions can make intent clearer:
token_weights = torch.tensor([1., 0.5, 0.25, 1., 0.])
weighted = hidden * token_weights[None, :, None]
print("weighted:", weighted.shape)

**Pitfall:** broadcasting can silently produce an unintended extra dimension. Assert output shapes after operations that combine differently shaped tensors.

## 5. Reductions, norms, and statistics

`sum`, `mean`, `max`, `softmax`, and vector norms all operate along selected dimensions. In Transformer code, `dim=-1` often means “operate across the hidden/features dimension.”

In [ ]:
x = torch.randn(2, 4, 8)

feature_mean = x.mean(dim=-1)
feature_std = x.std(dim=-1, unbiased=False)
feature_norm = torch.linalg.vector_norm(x, dim=-1)
probabilities = torch.softmax(x, dim=-1)

print("mean:", feature_mean.shape)
print("std:", feature_std.shape)
print("norm:", feature_norm.shape)
print("softmax:", probabilities.shape)
print("probability sums:", probabilities.sum(dim=-1))

## 6. Matrix multiplication and tensor contractions

Common operations:
- `@` / `torch.matmul`: matrix multiplication with batch dimensions.
- `torch.bmm`: batch matrix multiplication for 3-D tensors.
- `torch.einsum`: named-dimension tensor contraction.
- `torch.tensordot`: contraction across specified axes.

`einsum` is especially useful for expressing attention and multi-head operations clearly.

In [ ]:
# Batched matrix multiplication
A = torch.randn(2, 3, 4)
B = torch.randn(2, 4, 5)
C = torch.matmul(A, B)
print("matmul:", C.shape)

# Equivalent contraction with einsum
C_einsum = torch.einsum("bij,bjk->bik", A, B)
print("einsum:", C_einsum.shape, "close:", torch.allclose(C, C_einsum))

# Contract the final dimension of x with the first dimension of W
x = torch.randn(6, 4)
W = torch.randn(4, 7)
y = torch.einsum("bi,io->bo", x, W)
print("linear-like contraction:", y.shape)

# Part II — Autograd and safe parameter updates

## 7. Gradients and computational graphs

PyTorch records operations involving tensors with `requires_grad=True`. Calling `.backward()` computes gradients for leaf tensors.

Use:
- `optimizer.zero_grad(set_to_none=True)` before a training step.
- `loss.backward()` to compute gradients.
- `optimizer.step()` to update parameters.
- `torch.no_grad()` for inference or explicit parameter edits.

In [ ]:
w = torch.tensor([2.0, -1.0], requires_grad=True)
x = torch.tensor([3.0, 4.0])
target = torch.tensor(1.0)

prediction = (w * x).sum()
loss = (prediction - target) ** 2
loss.backward()

print("prediction:", prediction.item())
print("loss:", loss.item())
print("gradient:", w.grad)

## 8. In-place operations: when to avoid them

An in-place operation changes the existing tensor (`add_`, `mul_`, `copy_`, or `x += ...`). It can invalidate values autograd needs for gradient computation.

Prefer out-of-place expressions in model code unless memory profiling shows a meaningful need and the operation is safe.

In [ ]:
# Safe out-of-place activation transformation
x = torch.randn(3, 4, requires_grad=True)
y = x * 2.0
loss = y.square().mean()
loss.backward()
print("gradient exists:", x.grad is not None)

# For deliberate parameter edits, disable gradient tracking.
layer = nn.Linear(4, 3)
with torch.no_grad():
    layer.weight.mul_(0.9)

## 9. Gradient clipping and finite-value checks

In [ ]:
model = nn.Sequential(nn.Linear(10, 16), nn.GELU(), nn.Linear(16, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_x = torch.randn(32, 10)
batch_y = torch.randn(32, 1)

optimizer.zero_grad(set_to_none=True)
pred = model(batch_x)
loss = F.mse_loss(pred, batch_y)
loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print("loss:", float(loss.detach()))
print("pre-clip total grad norm:", float(grad_norm))
assert torch.isfinite(loss)

# Part III — Core neural-network tensor transformations

## 10. Linear transformations and embeddings

A linear layer transforms the final feature dimension. An embedding table maps integer token IDs to learned vectors.

In [ ]:
batch, seq_len, vocab_size, hidden_size = 2, 5, 100, 16
token_ids = torch.randint(0, vocab_size, (batch, seq_len))

embedding = nn.Embedding(vocab_size, hidden_size)
hidden = embedding(token_ids)

projection = nn.Linear(hidden_size, 32)
projected = projection(hidden)

print("token IDs:", token_ids.shape)
print("embedded:", hidden.shape)
print("projected:", projected.shape)

## 11. Activation functions and gating

Activations introduce non-linearity. GELU is common in Transformer feed-forward blocks; SiLU is used in many modern decoder architectures. Gating combines a content pathway with a learned gate.

In [ ]:
x = torch.linspace(-4, 4, 100)
activations = {
    "relu": F.relu(x),
    "gelu": F.gelu(x),
    "silu": F.silu(x),
    "tanh": torch.tanh(x),
}
{k: (float(v.min()), float(v.max())) for k, v in activations.items()}

In [ ]:
# A simple gated transformation
batch, seq, hidden_size = 2, 6, 16
h = torch.randn(batch, seq, hidden_size)
content = nn.Linear(hidden_size, hidden_size)(h)
gate = torch.sigmoid(nn.Linear(hidden_size, hidden_size)(h))
gated = content * gate
print("gated representation:", gated.shape)

## 12. Normalization: LayerNorm and RMSNorm concept

LayerNorm normalizes across a selected feature dimension and then applies learned scale and bias. RMSNorm scales by root-mean-square magnitude and commonly omits mean subtraction and bias.

Normalization stabilizes activation scales; it does not simply “make all values between zero and one.”

In [ ]:
x = torch.randn(2, 5, 16) * 3 + 2

layer_norm = nn.LayerNorm(16)
normalized = layer_norm(x)

rms = torch.sqrt(torch.mean(x.square(), dim=-1, keepdim=True) + 1e-6)
rms_normalized = x / rms

print("input mean/std:", x.mean().item(), x.std().item())
print("LayerNorm feature mean:", normalized.mean(dim=-1))
print("RMS-normalized RMS:", rms_normalized.square().mean(dim=-1).sqrt())

## 13. Convolution: local receptive-field transformations

Convolution applies a shared kernel across spatial or temporal positions. It is common in vision, audio, and time-series models. Tensor layout matters: PyTorch `Conv2d` expects `[batch, channels, height, width]`.

In [ ]:
images = torch.randn(4, 3, 32, 32)
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
feature_maps = conv(images)
print("images:", images.shape, "feature maps:", feature_maps.shape)

# Part IV — Transformer tensor mechanics

## 14. The Q, K, and V projections

Given hidden states \(X\in\mathbb{R}^{B\times T\times d_{model}}\), learned projections produce:

\[
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
\]

Attention compares query vectors with key vectors, then uses the resulting weights to combine value vectors.

For a single head:

\[
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

where \(M\) is an optional mask/bias. The scaling by \(\sqrt{d_k}\) controls score magnitude.

In [ ]:
B, T, d_model, d_k = 2, 5, 16, 8
X = torch.randn(B, T, d_model)
Wq = nn.Linear(d_model, d_k, bias=False)
Wk = nn.Linear(d_model, d_k, bias=False)
Wv = nn.Linear(d_model, d_k, bias=False)

Q, K, V = Wq(X), Wk(X), Wv(X)
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
weights = torch.softmax(scores, dim=-1)
context = weights @ V

print("Q/K/V:", Q.shape, K.shape, V.shape)
print("scores:", scores.shape, "weights:", weights.shape, "context:", context.shape)
print("attention row sums:", weights.sum(dim=-1))

## 15. Causal masks and padding masks

A causal mask prevents a token from attending to future tokens. A padding mask prevents attention to padding positions. They solve different problems and may be combined.

For `scaled_dot_product_attention`, boolean mask semantics are: `True` indicates positions allowed to participate. Other APIs may use the opposite convention, so check the function documentation.

In [ ]:
T = 5
# True on and below diagonal: query position may attend to current and earlier keys.
causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
print(causal_allowed.int())

# Example key padding mask: True means this token is real (not padding).
key_is_real = torch.tensor([
    [True, True, True, False, False],
    [True, True, True, True, False],
])
# Broadcast to [batch, 1, query, key]
padding_allowed = key_is_real[:, None, None, :]
combined_allowed = causal_allowed[None, None, :, :] & padding_allowed
print("combined mask:", combined_allowed.shape)

## 16. Multi-head attention: reshape, transpose, attend, merge

The usual shape flow is:

`[B, T, d_model]` → `[B, T, H, d_head]` → `[B, H, T, d_head]`

After attention, transpose and reshape back to `[B, T, d_model]`.

In [ ]:
def split_heads(x: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq_len, hidden = x.shape
    assert hidden % num_heads == 0
    head_dim = hidden // num_heads
    return x.reshape(batch, seq_len, num_heads, head_dim).transpose(1, 2)

def merge_heads(x: torch.Tensor) -> torch.Tensor:
    batch, heads, seq_len, head_dim = x.shape
    return x.transpose(1, 2).contiguous().reshape(batch, seq_len, heads * head_dim)

B, T, d_model, heads = 2, 6, 32, 4
z = torch.randn(B, T, d_model)
z_heads = split_heads(z, heads)
z_restored = merge_heads(z_heads)
print("split:", z_heads.shape, "merged:", z_restored.shape)
assert torch.equal(z, z_restored)

## 17. Build a small multi-head self-attention module

This educational implementation exposes intermediate tensors. Production implementations should usually use optimized attention kernels such as PyTorch scaled dot-product attention or a supported fused attention backend.

In [ ]:
class InspectableSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, causal=True, return_intermediates=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = split_heads(q, self.num_heads)
        k = split_heads(k, self.num_heads)
        v = split_heads(v, self.num_heads)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if causal:
            mask = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        attended = weights @ v
        merged = merge_heads(attended)
        output = self.out(merged)

        if return_intermediates:
            return output, {"q": q, "k": k, "v": v, "scores": scores,
                            "weights": weights, "head_output": attended}
        return output

attn = InspectableSelfAttention(d_model=32, num_heads=4)
sample = torch.randn(2, 6, 32)
out, internals = attn(sample, return_intermediates=True)
print("output:", out.shape)
for key, value in internals.items():
    print(key, tuple(value.shape))

## 18. Use PyTorch scaled dot-product attention

`torch.nn.functional.scaled_dot_product_attention` may dispatch to optimized kernels depending on device, dtype, shape, and installed PyTorch build. Its mask convention differs from some other APIs; verify it carefully.

In [ ]:
q = torch.randn(2, 4, 6, 8)
k = torch.randn(2, 4, 6, 8)
v = torch.randn(2, 4, 6, 8)

fast_path_output = F.scaled_dot_product_attention(
    q, k, v,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True,
)
print(fast_path_output.shape)

# Part V — Hidden states, residual streams, and activation interventions

## 19. Residual streams

A Transformer block commonly adds a sublayer's output back into its input:

\[
x_{l+1}=x_l+F(Norm(x_l))
\]

The residual stream carries information across layers. Attention and feed-forward blocks write updates into it. Exact ordering varies by architecture (pre-norm, post-norm, parallel residual, etc.).

In [ ]:
class TinyPreNormBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, expansion=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = InspectableSelfAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, expansion * d_model),
            nn.GELU(),
            nn.Linear(expansion * d_model, d_model),
        )

    def forward(self, x, return_parts=False):
        attn_update = self.attn(self.norm1(x))
        after_attn = x + attn_update
        mlp_update = self.mlp(self.norm2(after_attn))
        output = after_attn + mlp_update
        if return_parts:
            return output, {
                "residual_in": x,
                "attention_update": attn_update,
                "residual_after_attention": after_attn,
                "mlp_update": mlp_update,
                "residual_out": output,
            }
        return output

block = TinyPreNormBlock()
x = torch.randn(2, 6, 32)
y, parts = block(x, return_parts=True)
for name, tensor in parts.items():
    print(name, tensor.shape, "norm", tensor.norm().item())

## 20. Capture intermediate activations with forward hooks

Hooks let you inspect activations without rewriting the module's forward method. Always remove hooks when finished to avoid accumulating callbacks or retaining tensors in memory.

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 16),
    nn.GELU(),
    nn.Linear(16, 4),
)

captured = {}
def save_activation(name):
    def hook(module, inputs, output):
        captured[name] = output.detach().cpu()
    return hook

handles = [
    model[0].register_forward_hook(save_activation("linear1")),
    model[1].register_forward_hook(save_activation("gelu")),
]

_ = model(torch.randn(3, 8))
for name, activation in captured.items():
    print(name, activation.shape)

for handle in handles:
    handle.remove()

## 21. Activation intervention: scale or suppress selected features

Activation steering modifies an intermediate representation during a forward pass. This is different from changing the model's learned weights. Interventions can be useful for controlled experiments, but arbitrary edits can damage behavior or create misleading conclusions.

In [ ]:
# A simple feature intervention on a synthetic hidden-state tensor.
hidden = torch.randn(2, 5, 12)
edited = hidden.clone()

feature_index = 3
edited[..., feature_index] *= 0.0  # ablate one feature dimension
edited[..., 7] += 0.5              # add a small offset to another feature

print("original feature 3 norm:", hidden[..., feature_index].norm().item())
print("edited feature 3 norm:", edited[..., feature_index].norm().item())

### Add a controlled direction to an activation

A direction vector can be added to a selected token or all tokens. In real interpretability work, the direction should be estimated from a principled contrast set, normalized, and evaluated against controls.

In [ ]:
hidden = torch.randn(2, 5, 12)
direction = torch.randn(12)
direction = F.normalize(direction, dim=0)

strength = 0.2
steered = hidden + strength * direction.view(1, 1, -1)
print("delta norm:", (steered - hidden).norm().item())

# Part VI — Weight manipulation and model adaptation

## 22. Initialization and safe weight edits

Weight initialization affects signal and gradient propagation. PyTorch modules generally initialize parameters automatically, but custom layers should use initialization appropriate to the activation and architecture.

In [ ]:
linear = nn.Linear(128, 64)
nn.init.kaiming_uniform_(linear.weight, a=math.sqrt(5))
if linear.bias is not None:
    nn.init.zeros_(linear.bias)

print("weight mean/std:", linear.weight.mean().item(), linear.weight.std().item())

## 23. Low-rank weight updates (LoRA as one technique)

LoRA changes a layer's effective weight:

\[
W'=W+\\frac{\\alpha}{r}BA
\]

It does not directly manipulate every activation tensor. It learns a compact update to selected weight matrices. This is useful for parameter-efficient fine-tuning, not a universal substitute for tensor operations or architecture changes.

In [ ]:
d_in, d_out, rank, alpha = 16, 12, 3, 6
W = torch.randn(d_out, d_in)
A = torch.randn(rank, d_in) * 0.01
B = torch.zeros(d_out, rank)
delta_W = (alpha / rank) * (B @ A)
W_effective = W + delta_W
print("base:", W.shape, "delta:", delta_W.shape, "effective:", W_effective.shape)

## 24. Weight interpolation and model merging

A simple interpolation between two compatible parameter tensors is:

\[
W_{mix}=(1-\lambda)W_A+\lambda W_B
\]

This is only meaningful when tensors correspond to the same architecture, shape, and compatible parameterization. Merging independently trained models is not guaranteed to preserve either model's performance.

In [ ]:
torch.manual_seed(1)
weight_a = torch.randn(4, 4)
weight_b = torch.randn(4, 4)
lam = 0.25
mixed = (1 - lam) * weight_a + lam * weight_b
print(mixed.shape, "interpolation fraction:", lam)

# Part VII — Compression and tensor factorization

## 25. Low-rank matrix factorization with SVD

Singular Value Decomposition factors a matrix into orthogonal directions and singular values. Truncating small singular values gives a low-rank approximation.

This is a mathematical compression technique; it is not the same training procedure as LoRA.

In [ ]:
M = torch.randn(64, 48)
U, S, Vh = torch.linalg.svd(M, full_matrices=False)

def truncated_svd_reconstruction(U, S, Vh, rank):
    return (U[:, :rank] * S[:rank]) @ Vh[:rank, :]

for rank in (4, 8, 16, 32, 48):
    approximation = truncated_svd_reconstruction(U, S, Vh, rank)
    relative_error = (M - approximation).norm() / M.norm()
    print(f"rank={rank:2d}, relative Frobenius error={relative_error.item():.4f}")

## 26. Tensor decomposition concepts

For higher-order tensors, useful factorization families include:

- **CP decomposition:** represents a tensor as a sum of rank-one outer products.
- **Tucker decomposition:** a smaller core tensor multiplied by factor matrices along each mode.
- **Tensor Train (TT):** represents a high-dimensional tensor as a chain of smaller cores.

Applications include compressing large layers, scientific computing, and structured model parameterization. Decomposition quality depends on tensor structure and chosen rank; it is not automatically faster on every accelerator.

In [ ]:
# Demonstrate a simple rank-k approximation to a matrix (a 2-D tensor).
matrix = torch.randn(20, 15)
u, s, vh = torch.linalg.svd(matrix, full_matrices=False)
k = 5
low_rank = (u[:, :k] * s[:k]) @ vh[:k]
print("original parameters:", matrix.numel())
print("factor parameters:", u[:, :k].numel() + s[:k].numel() + vh[:k].numel())
print("relative error:", ((matrix - low_rank).norm() / matrix.norm()).item())

## 27. Pruning and sparsity

Pruning sets selected weights to zero. Unstructured sparsity can reduce parameter values but may not accelerate inference unless the hardware and kernels exploit the sparsity pattern. Structured pruning removes whole channels, heads, or blocks and is often easier to turn into real speedups.

In [ ]:
layer = nn.Linear(128, 64)
original_nonzero = torch.count_nonzero(layer.weight).item()

# Simple magnitude thresholding for demonstration only.
with torch.no_grad():
    threshold = torch.quantile(layer.weight.abs(), 0.30)
    layer.weight[layer.weight.abs() < threshold] = 0.0

new_nonzero = torch.count_nonzero(layer.weight).item()
print("original nonzero:", original_nonzero)
print("remaining nonzero:", new_nonzero)
print("sparsity:", 1 - new_nonzero / layer.weight.numel())

For a training-aware pruning workflow, consider PyTorch pruning utilities, iterative schedules, fine-tuning after pruning, and evaluation of the exported model. Avoid assuming that zeroing weights alone reduces latency.

## 28. Quantization: lower-precision representations

Quantization maps floating-point values to a lower-precision representation. It can reduce memory bandwidth and model size, but may introduce numerical error.

Common families:
- FP16 / BF16: lower-precision floating point.
- INT8: integer quantization, often with calibration or quantization-aware training.
- 4-bit formats: aggressive compression, often used for large-model inference or QLoRA training.

The example below demonstrates simple symmetric fake quantization, not a production quantizer.

In [ ]:
def fake_symmetric_quantize(x: torch.Tensor, bits: int = 8):
    qmax = 2 ** (bits - 1) - 1
    scale = x.abs().max().clamp_min(1e-8) / qmax
    q = torch.clamp(torch.round(x / scale), -qmax - 1, qmax)
    return q * scale, scale

x = torch.randn(1000)
x_hat, scale = fake_symmetric_quantize(x, bits=8)
print("scale:", scale.item())
print("MSE:", F.mse_loss(x_hat, x).item())
print("max abs error:", (x_hat - x).abs().max().item())

# Part VIII — Performance and numerical best practices

## 29. Dtype, mixed precision, and device placement

- FP32 is a robust baseline.
- FP16 has a narrower exponent range and may need gradient scaling.
- BF16 has a wider exponent range than FP16 and is often preferred when supported.
- Keep numerically sensitive reductions in appropriate precision.
- Avoid repeated CPU↔GPU transfers inside a hot loop.

In [ ]:
x = torch.randn(1024, 1024, device=device)
print("device:", x.device, "dtype:", x.dtype)

if device.type == "cuda":
    supported = torch.cuda.is_bf16_supported()
    print("BF16 supported:", supported)
else:
    print("CPU demonstration; accelerator precision options vary by build.")

## 30. Inference mode, no_grad, and autocast

`torch.inference_mode()` is optimized for inference and disables autograd bookkeeping more aggressively than `no_grad()`. Autocast selects suitable lower precision for supported operations.

In [ ]:
small_model = nn.Sequential(nn.Linear(32, 64), nn.GELU(), nn.Linear(64, 8)).to(device)
input_batch = torch.randn(16, 32, device=device)

small_model.eval()
with torch.inference_mode():
    prediction = small_model(input_batch)
print(prediction.shape)

## 31. Avoid repeated allocations and synchronize only when needed

Best practices:
- Reuse buffers when appropriate.
- Batch operations instead of Python loops over individual tokens.
- Profile before optimizing.
- GPU operations are asynchronous; timing needs synchronization.
- Do not call `.item()` repeatedly inside performance-critical GPU loops.

In [ ]:
import time

a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)

if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
c = a @ b
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
print(f"matrix multiply elapsed: {elapsed:.4f}s on {device}")

## 32. Numerical stability checklist

- Use stable library functions (`log_softmax`, `cross_entropy`) instead of manually composing unstable expressions.
- Clamp denominators where zero is possible.
- Check `torch.isfinite()` on losses and important activations during debugging.
- Normalize vectors with an epsilon when zero vectors are possible.
- Be careful with softmax over the intended dimension.
- Test extreme values and long sequences.

In [ ]:
logits = torch.tensor([[1000.0, 1001.0, 999.0]])
stable_log_probs = F.log_softmax(logits, dim=-1)
print(stable_log_probs)
assert torch.isfinite(stable_log_probs).all()

v = torch.zeros(3)
v_normalized = F.normalize(v, dim=0, eps=1e-8)
print(v_normalized)

# Part IX — Testing and practical exercises

## 33. Shape and invariant tests

Tests catch many tensor bugs before they become model-quality problems. Validate dimensions, dtype, device, finiteness, and mathematical invariants.

In [ ]:
def test_attention_shapes():
    module = InspectableSelfAttention(d_model=32, num_heads=4)
    x = torch.randn(2, 7, 32)
    y, info = module(x, return_intermediates=True)
    assert y.shape == x.shape
    assert info["weights"].shape == (2, 4, 7, 7)
    assert torch.allclose(
        info["weights"].sum(dim=-1),
        torch.ones(2, 4, 7),
        atol=1e-5,
    )

def test_split_merge_round_trip():
    x = torch.randn(3, 9, 48)
    assert torch.equal(x, merge_heads(split_heads(x, 6)))

test_attention_shapes()
test_split_merge_round_trip()
print("All tensor invariant tests passed.")

## 34. Exercises

Try these before looking at the solution cells.

1. **Broadcasting:** multiply a `[batch, sequence, hidden]` tensor by one scalar per batch item. Make the shape explicit.
2. **Contraction:** use `einsum` to implement a batched linear layer with input `[B,T,I]` and weights `[O,I]`.
3. **Attention:** add a key-padding mask to the educational attention module.
4. **Residual stream:** measure the norm of each sublayer update relative to the residual input.
5. **Intervention:** ablate one attention head and compare output norms.
6. **Compression:** compare SVD reconstruction error at several ranks.
7. **Quantization:** compare fake INT8 quantization error with FP16 casting error.
8. **Testing:** write a test ensuring causal attention assigns zero probability to future positions.

In [ ]:
# Exercise 1 solution: per-batch scalar scaling.
x = torch.randn(3, 5, 8)
batch_scale = torch.tensor([0.5, 1.0, 2.0])
scaled = x * batch_scale[:, None, None]
assert scaled.shape == x.shape

In [ ]:
# Exercise 2 solution: batched linear contraction.
B, T, I, O = 2, 4, 6, 3
x = torch.randn(B, T, I)
w = torch.randn(O, I)
y = torch.einsum("bti,oi->bto", x, w)
assert y.shape == (B, T, O)

In [ ]:
# Exercise 8 solution: future positions receive no probability.
module = InspectableSelfAttention(d_model=16, num_heads=4)
_, info = module(torch.randn(1, 5, 16), causal=True, return_intermediates=True)
future_mask = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
future_probs = info["weights"][0, :, future_mask]
assert torch.allclose(future_probs, torch.zeros_like(future_probs), atol=1e-7)
print("Causal mask test passed.")

# Final reference — choosing the right operation

| Goal | Techniques to investigate |
|---|---|
| Change tensor shape/layout | `reshape`, `permute`, `transpose`, `contiguous` |
| Combine feature dimensions | Broadcasting, `matmul`, `einsum` |
| Learn representations | Embeddings, linear layers, convolutions |
| Stabilize activations | LayerNorm, RMSNorm, residual connections |
| Change token interactions | Attention projections, masks, Q/K/V, head operations |
| Inspect internal computation | Forward hooks, hidden states, activation statistics |
| Intervene on runtime representations | Activation ablation, scaling, steering directions |
| Adapt pretrained weights efficiently | LoRA, QLoRA, AdaLoRA, DoRA, VeRA |
| Compress parameters | SVD, CP/Tucker/TT decomposition, pruning, quantization |
| Improve execution | Fused attention, batching, mixed precision, profiling |

## The most important engineering rule

First identify **what you intend to change**:

- Shape or layout?
- Tensor values?
- A hidden-state activation during a forward pass?
- A learned parameter?
- The architecture or computation graph?

Then select the smallest, testable intervention that achieves the goal. Measure correctness, quality, memory, and latency before scaling it up.